# What \"Generative\" Means: Autoregression, the Chain Rule, and Perplexity

The full code of [What \"Generative\" Means: Autoregression, the Chain Rule, and Perplexity](https://residual-stream.lvtan71.workers.dev/posts/autoregression-and-perplexity/), run from [its folder](https://github.com/lvtan71/residual-stream-code/tree/main/posts/autoregression-and-perplexity) in the
order the folder's README gives. Each script is followed by the output it
printed for the post, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/autoregression-and-perplexity.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 22 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
%pip install -q "torch==2.14.*" tokenizers

# Setup: fetch the companion repo and work in this post's folder.
import os, subprocess, sys
SLUG = "autoregression-and-perplexity"
ROOT = os.path.abspath("..") if os.path.isdir("../posts") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
os.chdir(os.path.join(ROOT, "posts", SLUG))
print("working in", os.getcwd())

## `bigram.py`

The chain rule on "the cache" with a character bigram model counted on the training posts (add-one smoothing), and the bigram model's perplexity on the held-out posts.

In [ ]:
!{sys.executable} bigram.py

The post printed:

```text
p('t') = 0.0635
p('h' | 't') = 0.2066
p('e' | 'h') = 0.4248
p(' ' | 'e') = 0.2166
p('c' | ' ') = 0.0259
p('a' | 'c') = 0.1363
p('c' | 'a') = 0.0674
p('h' | 'c') = 0.1844
p('e' | 'h') = 0.4248
p('the cache') = product = 2.250e-08;  -log2 p = 25.4 bits for 9 characters
bigram model on held-out posts: 2.849 nats per character, perplexity 17.3, 4.110 bits per character
```

## `train_all.py`

Trains the LSTM language model of `lm.py` over characters and over byte-level BPE tokens, two seeds each, and prints held-out bits per byte at five checkpoints. The character model with seed 0 is saved as `char_lstm.pt`.

In [ ]:
!{sys.executable} train_all.py

The post printed:

```text
char seed 0: 148 units, 377,236 parameters, 1.01 bytes per unit, 0.2% of held-out units unseen in training; at 3,000 steps (41 passes) perplexity 6.4 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 3.311 / 2.973 / 2.737 / 2.621 / 2.657
char seed 1: 148 units, 377,236 parameters, 1.01 bytes per unit, 0.2% of held-out units unseen in training; at 3,000 steps (41 passes) perplexity 6.4 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 3.297 / 2.967 / 2.741 / 2.647 / 2.659
bpe  seed 0: 2,048 units, 987,136 parameters, 2.56 bytes per unit, 0.6% of held-out units unseen in training; at 3,000 steps (113 passes) perplexity 261.5 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 2.912 / 2.730 / 2.692 / 2.853 / 3.133
bpe  seed 1: 2,048 units, 987,136 parameters, 2.56 bytes per unit, 0.6% of held-out units unseen in training; at 3,000 steps (113 passes) perplexity 268.9 per unit
      validation bits per byte at 250 / 500 / 1,000 / 2,000 / 3,000 steps: 2.903 / 2.728 / 2.690 / 2.865 / 3.148
```

## `generate.py`

Greedy and sampled continuations of "The KV cache " from `char_lstm.pt`, and the time to score 512 known characters in one call, one call per character, and to generate 512. Reads `char_lstm.pt`.

In [ ]:
!{sys.executable} generate.py

The post printed:

```text
greedy:  The KV cache 'post costs a settings of the same post is the same post is the same post is the same post is the same post is the same post is the same post 2.1 on the same post 2.1 found the same post 2.1 on the sam'
sampled: The KV cache 'han the routure model. Write at it\n*sort** (Zheng](hts forkens only a\nmeany sublay.\n- Imm and generated units including token\n  to best process to the graming story, and every step. The loss over\n  us'
512 characters: teacher forcing 5.9 ms; the same known characters one call at a time 16.4 ms; generating 19.2 ms
```

## `baselines.py`

Bits per byte on the held-out posts for raw bytes, a uniform guess, unigram counts, and zlib, bz2 and xz with and without the training posts.

In [ ]:
!{sys.executable} baselines.py

The post printed:

```text
raw UTF-8 bytes:                         8.00 bits per byte
uniform guess over 148 characters:        7.13
unigram character counts:                4.95
zlib -9  on the held-out posts alone 3.15; given the training posts 3.03
bz2 -9   on the held-out posts alone 2.87; given the training posts 2.34
xz -9e   on the held-out posts alone 2.95; given the training posts 2.44
```